In [1]:
import sqlite3
import pandas as pd

# One file holds the whole database
conn = sqlite3.connect("../data/ridgeline_screen.db")

review = pd.read_csv("../data/processed/universe_review.csv")
metrics = pd.read_csv("../data/processed/metrics.csv")
gate = pd.read_csv("../data/processed/size_gate_results.csv")
valuation = pd.read_csv("../data/processed/valuation.csv")
prices = pd.read_csv("../data/raw/share_prices.csv")

# companies: one row per company
companies = review[["cik", "ticker", "company_name", "sic_code", "sic_description",
                    "fiscal_year_end", "hq_state", "in_universe", "exclusion_reason"]]

# financials: one row per company and year (drop helper columns used only in calculations)
keep = [c for c in metrics.columns
        if c not in ("ticker", "acquisitions_paid_filled") and "_lag" not in c]
financials = metrics[keep]

# size_gate: one row per tested company
size_gate = gate[["cik", "fiscal_year", "size_gate"]]

# market_data: one row per company and price date
market_data = (valuation[["cik", "ticker", "close_price", "shares_used_000s", "shares_basis", "shares_as_of"]]
               .merge(prices[["ticker", "price_date", "price_source"]], on="ticker")
               .drop(columns="ticker"))

for name, table in {"companies": companies, "financials": financials,
                    "size_gate": size_gate, "market_data": market_data}.items():
    table.to_sql(name, conn, if_exists="replace", index=False)
    print(f"Saved table {name:12} {len(table):>6} rows")

Saved table companies       182 rows
Saved table financials     1640 rows
Saved table size_gate       125 rows
Saved table market_data       8 rows


In [2]:
query = """
SELECT c.ticker,
       c.company_name,
       f.fiscal_year,
       ROUND(f.revenue / 1000, 1)      AS revenue_m,
       ROUND(f.ebitda_margin * 100, 1) AS ebitda_margin_pct
FROM financials AS f
JOIN companies AS c ON c.cik = f.cik
WHERE c.ticker = 'MG'
ORDER BY f.fiscal_year DESC
LIMIT 5;
"""
pd.read_sql(query, conn)

,ticker,company_name,fiscal_year,revenue_m,ebitda_margin_pct
0,MG,"Mistras Group, Inc.",2025,724.0,9.9
1,MG,"Mistras Group, Inc.",2024,729.6,10.0
2,MG,"Mistras Group, Inc.",2023,705.5,4.6
3,MG,"Mistras Group, Inc.",2022,687.4,7.7
4,MG,"Mistras Group, Inc.",2021,677.1,7.8
